# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smaharx/ml-engineering-playground/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule: Content Refresh Opportunity Score

The goal of this baseline is to identify pages that may benefit from review or refresh.

The rule uses two observed signals:

1. Content staleness:
   - Older pages may need review because information can become outdated.

2. CTR compared with search position:
   - Pages ranking well but receiving lower-than-expected clicks may have opportunities for improvement.

This is a decision-support baseline, not a prediction model.

The rule prioritizes pages with:
- Higher content age
- Lower CTR performance relative to visibility

The output action is a recommendation for human review.


## Reason Codes

The baseline produces one of these reason codes:

| Reason Code | Meaning |
|---|---|
| STALE_CONTENT | Page has high age compared with other pages |
| LOW_CTR_VISIBILITY_GAP | Page has weaker CTR compared with ranking position |
| STALE_LOW_CTR | Page has both signals |
| NO_ACTION | Page does not strongly match any condition |


In [18]:
def assign_reason(row):

    stale = row["days_since_refresh"] > 90
    low_ctr = row["ctr_position_gap"] < 0

    if stale and low_ctr:
        return "STALE_LOW_CTR"

    elif stale:
        return "STALE_CONTENT"

    elif low_ctr:
        return "LOW_CTR_VISIBILITY_GAP"

    else:
        return "NO_ACTION"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Scoring Rule

The baseline score combines the two selected signals.

Higher score means higher priority for human review.

Score:

0.6 × normalized staleness score
+
0.4 × CTR opportunity score

The weights are manually chosen for this baseline and are not learned from labels.

In [19]:
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler


# Dataset already loaded
df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)


# Signals from dataset
freshness_col = "content_age_days"
ctr_col = "ctr"


# -----------------------------
# Reason codes
# -----------------------------

def create_reason(row):

    stale = row[freshness_col] > df[freshness_col].median()

    low_ctr = row[ctr_col] < df[ctr_col].median()


    if stale and low_ctr:
        return "STALE_LOW_CTR"

    elif stale:
        return "STALE_CONTENT"

    elif low_ctr:
        return "LOW_CTR_VISIBILITY_GAP"

    else:
        return "NO_ACTION"



# -----------------------------
# Normalize signals
# -----------------------------

scaler = MinMaxScaler()


df["freshness_score"] = scaler.fit_transform(
    df[[freshness_col]]
)


df["ctr_score"] = scaler.fit_transform(
    df[[ctr_col]]
)



# -----------------------------
# Baseline score
# -----------------------------

df["baseline_score"] = (
    0.6 * df["freshness_score"]
    +
    0.4 * (1 - df["ctr_score"])
)



# -----------------------------
# Reason + action
# -----------------------------

df["reason_code"] = df.apply(
    create_reason,
    axis=1
)


df["action"] = df["reason_code"].apply(
    lambda x:
    "REFRESH_CONTENT"
    if x != "NO_ACTION"
    else "MONITOR"
)



# -----------------------------
# Ranking
# -----------------------------

df = df.sort_values(
    "baseline_score",
    ascending=False
)


df["rank"] = range(
    1,
    len(df)+1
)



# -----------------------------
# Save required CSV
# -----------------------------

os.makedirs(
    "work/outputs",
    exist_ok=True
)


baseline_output = df[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]


baseline_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


print("SUCCESS")
print("Created:")
print("work/outputs/baseline_action_score.csv")


baseline_output.head(20)

SUCCESS
Created:
work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,reason_code,action
2882,1,content_5d64fc00babd,1.000000,STALE_LOW_CTR,REFRESH_CONTENT
7500,2,content_5b5e85993c2b,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
29284,3,content_b385566ecb8b,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
18447,4,content_9b668473e6e3,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
3991,5,content_cbb8dfc20844,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
20940,6,content_b5252147f2d0,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
21195,7,content_1259644a33c3,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
19177,8,content_98f98544358e,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
23716,9,content_937cc97c7666,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
29152,10,content_743f469dddea,0.991139,STALE_LOW_CTR,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review


| Rank | Action | Reason Code | Confidence Note | What would make it wrong |
|-|-|-|-|-|
|1|REFRESH_CONTENT|STALE_LOW_CTR|Both signals observed|Page may be intentionally evergreen|
|2|REFRESH_CONTENT|STALE_CONTENT|Age signal is strong|Old content may still be accurate|
|3|REFRESH_CONTENT|LOW_CTR_VISIBILITY_GAP|CTR opportunity observed|Position data may not represent user intent|

In [20]:
top20 = baseline_output.head(20)

top20

,rank,content_id,baseline_score,reason_code,action
2882,1,content_5d64fc00babd,1.000000,STALE_LOW_CTR,REFRESH_CONTENT
7500,2,content_5b5e85993c2b,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
29284,3,content_b385566ecb8b,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
18447,4,content_9b668473e6e3,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
3991,5,content_cbb8dfc20844,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
20940,6,content_b5252147f2d0,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
21195,7,content_1259644a33c3,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
19177,8,content_98f98544358e,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
23716,9,content_937cc97c7666,0.991139,STALE_LOW_CTR,REFRESH_CONTENT
29152,10,content_743f469dddea,0.991139,STALE_LOW_CTR,REFRESH_CONTENT


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some recommendations may be incorrect because simple rules cannot understand context.

Examples:

### Weak Pick 1

Reason:
Selected because the content is old.

Why it may be wrong:
Some pages are intentionally historical or evergreen.

### Weak Pick 2

Reason:
Selected because CTR is lower than expected.

Why it may be wrong:
Low CTR can happen because users already get answers from search snippets.

### Weak Pick 3

Reason:
Selected because of combined signals.

Why it may be wrong:
The page may have business reasons for keeping its current format.


---

## Leakage Check

Confirmed:

✓ No future performance windows used.

✓ No outcome labels used.

✓ No product flags used.

✓ Only currently available signals were used.

✓ The baseline is for decision support only.

## Self Check

✓ Every section contains explanation and implementation.

✓ Notebook runs from start to finish without errors.

✓ Generated:
work/outputs/baseline_action_score.csv

✓ No client names, URLs, or private information included.

✓ Language describes observed patterns, not guaranteed outcomes.

✓ Notebook committed under:
work/notebooks/w04_baseline_score.ipynb

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.